# Modellvergleich über Kurswochen

Für jeden Stichtag vergleichen wir:

1. Referenz: Kurs und Anmeldetag
2. Boosting: Kurs, Anmeldung, VLE-Aktivität und Assessments

Alle Merkmale stammen aus dem jeweiligen Datenstand. Personen bleiben
über sämtliche Wochen hinweg im selben Validierungs-Fold.

Wir vergleichen Average Precision (AP) mit dem Zielanteil der jeweiligen
Woche. Für die praktische Nutzung messen wir außerdem Recall und
Trefferquote bei 10 % und 20 % Hinweisen je Kurspräsentation.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

projektordner = Path.cwd()
if projektordner.name.lower() == "notebooks":
    projektordner = projektordner.parent

datei = (
    projektordner
    / "data"
    / "processed"
    / "oulad_wochenmodellbasis.csv"
)
wochen = pd.read_csv(datei)

# Die Wochen werden zunächst mit früheren Präsentationen verglichen.
entwicklung = wochen.loc[
    wochen["code_presentation"].ne("2014J")
].copy()

# Eine Person erhält genau einen Fold. Bei mehreren Kurseinträgen
# dient "mindestens ein späterer Fall" nur zur ausgewogenen Aufteilung.
personen = (
    entwicklung.groupby("id_student", as_index=False)[
        "abbruch_oder_fail"
    ]
    .max()
)
personen["fold"] = -1

aufteilung = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)

for fold, (_, valid_idx) in enumerate(
    aufteilung.split(
        personen[["id_student"]],
        personen["abbruch_oder_fail"],
    )
):
    personen.loc[valid_idx, "fold"] = fold

fold_je_person = dict(
    zip(personen["id_student"], personen["fold"])
)
entwicklung["fold"] = (
    entwicklung["id_student"]
    .map(fold_je_person)
    .astype(int)
)

assert entwicklung["fold"].between(0, 2).all()
assert entwicklung["stichtag"].nunique() == 9

print("Personen in der Entwicklung:", len(personen))
print("Datenstände in der Entwicklung:", len(entwicklung))
print(
    "Stichtage:",
    sorted(entwicklung["stichtag"].unique().tolist()),
)

Personen in der Entwicklung: 17762
Datenstände in der Entwicklung: 156973
Stichtage: [6, 13, 20, 27, 55, 83, 111, 167, 209]


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

kurs = ["code_module", "code_presentation"]

referenz_merkmale = kurs + ["anmeldetag"]

alle_merkmale = referenz_merkmale + [
    "assessments_faellig",
    "abgaben",
    "assessments_fehlend",
    "banked_faellige",
    "klicks_bis_stichtag",
    "klicks_letzte_7_tage",
]

# assessments_erfuellt wird nicht zusätzlich verwendet:
# assessments_fehlend ergibt sich bereits aus fällig minus erfüllt.
def erstelle_modell(name, merkmale):
    """Erstellt Vorverarbeitung und Klassifikator für einen Fold."""
    zahlen = [
        spalte for spalte in merkmale
        if spalte not in kurs
    ]

    vorbereitung = ColumnTransformer(
        transformers=[
            (
                "kurs",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
                kurs,
            ),
            ("zahlen", StandardScaler(), zahlen),
        ]
    )

    if name == "Referenz":
        klassifikator = LogisticRegression(max_iter=1000)
    else:
        klassifikator = HistGradientBoostingClassifier(
            random_state=42
        )

    return Pipeline([
        ("vorbereitung", vorbereitung),
        ("modell", klassifikator),
    ])


def hinweise_je_kurs(daten, risiken, anteil):
    """Bewertet die höchsten Risiken innerhalb jeder Präsentation."""
    rangliste = daten[
        kurs + ["id_student", "abbruch_oder_fail"]
    ].copy()
    rangliste["risiko"] = risiken

    rangliste = rangliste.sort_values(
        kurs + ["risiko"],
        ascending=[True, True, False],
        kind="mergesort",
    )

    rangliste["rang"] = rangliste.groupby(kurs).cumcount()
    kursgroesse = rangliste.groupby(kurs)[
        "id_student"
    ].transform("size")

    kontingent = np.ceil(kursgroesse * anteil / 100)
    mit_hinweis = rangliste["rang"] < kontingent

    anzahl_hinweise = int(mit_hinweis.sum())
    treffer = int(
        rangliste.loc[
            mit_hinweis, "abbruch_oder_fail"
        ].sum()
    )
    alle_faelle = int(
        rangliste["abbruch_oder_fail"].sum()
    )

    return {
        f"hinweise_{anteil}": anzahl_hinweise,
        f"treffer_{anteil}": treffer,
        f"recall_{anteil}": treffer / alle_faelle,
        f"trefferquote_{anteil}": treffer / anzahl_hinweise,
    }

In [3]:
ergebnisse = []

varianten = {
    "Referenz": referenz_merkmale,
    "Boosting": alle_merkmale,
}

for tag in sorted(entwicklung["stichtag"].unique()):
    daten = (
        entwicklung.loc[
            entwicklung["stichtag"].eq(tag)
        ]
        .reset_index(drop=True)
    )

    ziel = daten["abbruch_oder_fail"].astype(int).to_numpy()
    zielanteil = ziel.mean()

    for name, merkmale in varianten.items():
        # Jede Zeile erhält eine Vorhersage von einem Modell,
        # das diese Person nicht im Training gesehen hat.
        risiken = np.full(len(daten), np.nan)

        for fold in range(3):
            train_idx = np.flatnonzero(
                daten["fold"].to_numpy() != fold
            )
            valid_idx = np.flatnonzero(
                daten["fold"].to_numpy() == fold
            )

            modell = erstelle_modell(name, merkmale)
            modell.fit(
                daten.iloc[train_idx][merkmale],
                ziel[train_idx],
            )

            risiken[valid_idx] = modell.predict_proba(
                daten.iloc[valid_idx][merkmale]
            )[:, 1]

        assert np.isfinite(risiken).all()

        ap = average_precision_score(ziel, risiken)

        zeile = {
            "stichtag": tag,
            "verfahren": name,
            "kurseintraege": len(daten),
            "faelle": int(ziel.sum()),
            "zielanteil": zielanteil,
            "AP": ap,
            "AP_zu_zielanteil": ap / zielanteil,
        }

        for anteil in (10, 20):
            zeile.update(
                hinweise_je_kurs(daten, risiken, anteil)
            )

        ergebnisse.append(zeile)
        print("Fertig:", tag, name)

vergleich_wochen = pd.DataFrame(ergebnisse)
display(vergleich_wochen.round(3))

Fertig: 6 Referenz
Fertig: 6 Boosting
Fertig: 13 Referenz
Fertig: 13 Boosting
Fertig: 20 Referenz
Fertig: 20 Boosting
Fertig: 27 Referenz
Fertig: 27 Boosting
Fertig: 55 Referenz
Fertig: 55 Boosting
Fertig: 83 Referenz
Fertig: 83 Boosting
Fertig: 111 Referenz
Fertig: 111 Boosting
Fertig: 167 Referenz
Fertig: 167 Boosting
Fertig: 209 Referenz
Fertig: 209 Boosting


,stichtag,verfahren,kurseintraege,faelle,zielanteil,AP,AP_zu_zielanteil,hinweise_10,treffer_10,recall_10,trefferquote_10,hinweise_20,treffer_20,recall_20,trefferquote_20
0,6,Referenz,18934,9025,0.477,0.539,1.130,1901,865,0.096,0.455,3794,1702,0.189,0.449
1,6,Boosting,18934,9025,0.477,0.640,1.343,1901,1316,0.146,0.692,3794,2542,0.282,0.670
2,13,Referenz,18724,8809,0.470,0.531,1.129,1879,848,0.096,0.451,3751,1658,0.188,0.442
3,13,Boosting,18724,8809,0.470,0.679,1.443,1879,1471,0.167,0.783,3751,2713,0.308,0.723
4,20,Referenz,18583,8664,0.466,0.524,1.124,1863,840,0.097,0.451,3720,1635,0.189,0.440
5,20,Boosting,18583,8664,0.466,0.718,1.539,1863,1588,0.183,0.852,3720,2855,0.330,0.767
6,27,Referenz,18309,8386,0.458,0.512,1.118,1836,816,0.097,0.444,3667,1585,0.189,0.432
7,27,Boosting,18309,8386,0.458,0.716,1.563,1836,1608,0.192,0.876,3667,2801,0.334,0.764
8,55,Referenz,17632,7707,0.437,0.486,1.112,1769,764,0.099,0.432,3531,1498,0.194,0.424
9,55,Boosting,17632,7707,0.437,0.764,1.747,1769,1654,0.215,0.935,3531,2958,0.384,0.838


## Vorlauf der tatsächlichen Hinweise

Die bisherigen Kennzahlen zeigen, wie viele spätere Fälle das Modell
findet. Nun speichern wir die Vorhersagen je Kurseintrag und Stichtag.

Das spätere Kursergebnis und der Abmeldetag werden ausschließlich für
diese nachträgliche Auswertung ergänzt. Sie sind keine Modellmerkmale.

In [4]:
schluessel = ["code_module", "code_presentation", "id_student"]
oof_teile = []

for tag in sorted(entwicklung["stichtag"].unique()):
    daten = (
        entwicklung.loc[entwicklung["stichtag"].eq(tag)]
        .reset_index(drop=True)
    )
    ziel = daten["abbruch_oder_fail"].astype(int).to_numpy()
    risiken = np.full(len(daten), np.nan)

    # Jede Person erhält eine Vorhersage aus einem Modell,
    # dessen Training diese Person nicht enthalten hat.
    for fold in range(3):
        train_idx = np.flatnonzero(
            daten["fold"].to_numpy() != fold
        )
        valid_idx = np.flatnonzero(
            daten["fold"].to_numpy() == fold
        )

        modell = erstelle_modell("Boosting", alle_merkmale)
        modell.fit(
            daten.iloc[train_idx][alle_merkmale],
            ziel[train_idx],
        )
        risiken[valid_idx] = modell.predict_proba(
            daten.iloc[valid_idx][alle_merkmale]
        )[:, 1]

    assert np.isfinite(risiken).all()

    teil = daten[
        schluessel + ["stichtag", "abbruch_oder_fail"]
    ].copy()
    teil["risiko_oof"] = risiken
    oof_teile.append(teil)

    print("Vorhersagen gespeichert für Tag", tag)

oof_wochen = pd.concat(oof_teile, ignore_index=True)

# Die Auswahl erfolgt getrennt innerhalb jeder Kurspräsentation.
gruppen = ["stichtag", "code_module", "code_presentation"]
oof_wochen = oof_wochen.sort_values(
    gruppen + ["risiko_oof"],
    ascending=[True, True, True, False],
    kind="mergesort",
).reset_index(drop=True)

oof_wochen["rang_im_kurs"] = (
    oof_wochen.groupby(gruppen).cumcount()
)
kursgroesse = (
    oof_wochen.groupby(gruppen)["id_student"]
    .transform("size")
)

for anteil in (10, 20):
    oof_wochen[f"hinweis_{anteil}"] = (
        oof_wochen["rang_im_kurs"]
        < np.ceil(kursgroesse * anteil / 100)
    )

# Kontrolle: dieselbe Zahl von Hinweisen wie im Modellvergleich.
erwartet = (
    vergleich_wochen.loc[
        vergleich_wochen["verfahren"].eq("Boosting")
    ]
    .set_index("stichtag")["hinweise_10"]
    .sort_index()
    .to_numpy()
)
beobachtet = (
    oof_wochen.groupby("stichtag")["hinweis_10"]
    .sum()
    .sort_index()
    .to_numpy()
)
assert np.array_equal(beobachtet, erwartet)

datei_risiken = (
    projektordner
    / "data"
    / "processed"
    / "oulad_wochen_oof_risiken.csv"
)
oof_wochen.to_csv(datei_risiken, index=False)

print("Gespeichert:", datei_risiken)
print("Datenstände:", len(oof_wochen))

Vorhersagen gespeichert für Tag 6
Vorhersagen gespeichert für Tag 13
Vorhersagen gespeichert für Tag 20
Vorhersagen gespeichert für Tag 27
Vorhersagen gespeichert für Tag 55
Vorhersagen gespeichert für Tag 83
Vorhersagen gespeichert für Tag 111
Vorhersagen gespeichert für Tag 167
Vorhersagen gespeichert für Tag 209
Gespeichert: c:\Users\Tony Willert\OneDrive\Weiterbildung\PortfolioProjekt\data\processed\oulad_wochen_oof_risiken.csv
Datenstände: 156973


In [5]:
rohordner = projektordner / "Daten OULAD"

info_ergebnis = pd.read_csv(
    rohordner / "studentInfo.csv",
    usecols=schluessel + ["final_result"],
)
abmeldungen = pd.read_csv(
    rohordner / "studentRegistration.csv",
    usecols=schluessel + ["date_unregistration"],
)

details = (
    oof_wochen
    .merge(
        info_ergebnis,
        on=schluessel,
        validate="many_to_one",
    )
    .merge(
        abmeldungen,
        on=schluessel,
        validate="many_to_one",
    )
)
details["abmeldetag"] = pd.to_numeric(
    details["date_unregistration"],
    errors="coerce",
)

# Das spätere Ergebnis muss zum bereits gespeicherten Ziel passen.
assert (
    details["final_result"].isin(["Withdrawn", "Fail"])
    == details["abbruch_oder_fail"]
).all()

auswertung_vorlauf = []

for tag in sorted(details["stichtag"].unique()):
    am_tag = details.loc[details["stichtag"].eq(tag)]

    for anteil in (10, 20):
        hinweise = am_tag.loc[am_tag[f"hinweis_{anteil}"]]
        abbruch_treffer = hinweise.loc[
            hinweise["final_result"].eq("Withdrawn")
        ]
        fail_treffer = hinweise.loc[
            hinweise["final_result"].eq("Fail")
        ]

        vorlauf = abbruch_treffer["abmeldetag"] - tag

        auswertung_vorlauf.append({
            "stichtag": tag,
            "hinweisquote": anteil,
            "hinweise": len(hinweise),
            "abbruch_treffer": len(abbruch_treffer),
            "fail_treffer": len(fail_treffer),
            "median_vorlauf_abbruch_tage": (
                round(vorlauf.median(), 1)
            ),
            "abbruch_innerhalb_14_tagen": int(
                vorlauf.le(14).sum()
            ),
        })

vorlauf_tabelle = pd.DataFrame(auswertung_vorlauf)
display(vorlauf_tabelle)

,stichtag,hinweisquote,hinweise,abbruch_treffer,fail_treffer,median_vorlauf_abbruch_tage,abbruch_innerhalb_14_tagen
0,6,10,1901,566,750,63.0,66
1,6,20,3794,1108,1434,66.0,118
2,13,10,1879,577,894,63.0,115
3,13,20,3751,1104,1609,66.0,206
4,20,10,1863,649,939,45.0,182
5,20,20,3720,1146,1709,53.0,282
6,27,10,1836,592,1016,52.0,137
7,27,20,3667,1064,1737,62.0,200
8,55,10,1769,443,1211,49.0,100
9,55,20,3531,887,2071,55.0,174


## Zeitliche Prüfung auf 2014J

Für jeden Stichtag trainieren wir mit früheren Kurspräsentationen
und prüfen die Vorhersagen auf 2014J. Personen, die in beiden
Zeiträumen vorkommen, werden aus der Prüfung ausgeschlossen.

Die Ergebnisse sind eine zeitliche Belastungsprobe. 2014J wurde
im Projekt bereits für frühere Analysen betrachtet und ist daher
kein unangetasteter Abschlusstest.

In [ ]:
# Personen, die in früheren Präsentationen vorkamen,
# dürfen nicht zugleich im zeitlichen Prüfteil stehen.
fruehere_personen = set(entwicklung["id_student"])

alle_2014j = wochen.loc[
    wochen["code_presentation"].eq("2014J")
].copy()

ueberschneidende_personen = (
    set(alle_2014j["id_student"]) & fruehere_personen
)

pruefung_2014j = alle_2014j.loc[
    ~alle_2014j["id_student"].isin(fruehere_personen)
].copy()

assert set(pruefung_2014j["id_student"]).isdisjoint(
    fruehere_personen
)

print(
    "Wegen Personenüberschneidung ausgeschlossen:",
    len(ueberschneidende_personen),
)
print(
    "Personen in der zeitlichen Prüfung:",
    pruefung_2014j["id_student"].nunique(),
)

zeitliche_ergebnisse = []
zeitliche_risiken = []

for tag in sorted(entwicklung["stichtag"].unique()):
    train = entwicklung.loc[
        entwicklung["stichtag"].eq(tag)
    ]
    test = pruefung_2014j.loc[
        pruefung_2014j["stichtag"].eq(tag)
    ]

    ziel_train = train["abbruch_oder_fail"].astype(int)
    ziel_test = test["abbruch_oder_fail"].astype(int)

    for name, merkmale in varianten.items():
        modell = erstelle_modell(name, merkmale)
        modell.fit(train[merkmale], ziel_train)

        risiko = modell.predict_proba(
            test[merkmale]
        )[:, 1]

        ap = average_precision_score(ziel_test, risiko)

        zeile = {
            "stichtag": tag,
            "verfahren": name,
            "kurseintraege": len(test),
            "faelle": int(ziel_test.sum()),
            "zielanteil": ziel_test.mean(),
            "AP": ap,
            "AP_zu_zielanteil": ap / ziel_test.mean(),
        }

        for anteil in (10, 20):
            zeile.update(
                hinweise_je_kurs(test, risiko, anteil)
            )

        zeitliche_ergebnisse.append(zeile)

        # Einzelne Boosting-Vorhersagen für die spätere
        # Vorlaufanalyse behalten, ohne erneut zu trainieren.
        if name == "Boosting":
            teil = test[
                schluessel + [
                    "stichtag",
                    "abbruch_oder_fail",
                ]
            ].copy()
            teil["risiko"] = risiko
            zeitliche_risiken.append(teil)

        print("Fertig:", tag, name)

vergleich_2014j = pd.DataFrame(zeitliche_ergebnisse)
risiken_2014j = pd.concat(
    zeitliche_risiken,
    ignore_index=True,
)

display(vergleich_2014j.round(3)) # pyright: ignore[reportUndefinedVariable]

Wegen Personenüberschneidung ausgeschlossen: 1403
Personen in der zeitlichen Prüfung: 8269
Fertig: 6 Referenz
Fertig: 6 Boosting
Fertig: 13 Referenz
Fertig: 13 Boosting
Fertig: 20 Referenz
Fertig: 20 Boosting
Fertig: 27 Referenz
Fertig: 27 Boosting
Fertig: 55 Referenz
Fertig: 55 Boosting
Fertig: 83 Referenz
Fertig: 83 Boosting
Fertig: 111 Referenz
Fertig: 111 Boosting
Fertig: 167 Referenz
Fertig: 167 Boosting
Fertig: 209 Referenz
Fertig: 209 Boosting


,stichtag,verfahren,kurseintraege,faelle,zielanteil,AP,AP_zu_zielanteil,hinweise_10,treffer_10,recall_10,trefferquote_10,hinweise_20,treffer_20,recall_20,trefferquote_20
0,6,Referenz,8644,3782,0.438,0.493,1.126,867,398,0.105,0.459,1731,803,0.212,0.464
1,6,Boosting,8644,3782,0.438,0.633,1.447,867,629,0.166,0.725,1731,1167,0.309,0.674
2,13,Referenz,7910,3039,0.384,0.440,1.145,794,328,0.108,0.413,1585,663,0.218,0.418
3,13,Boosting,7910,3039,0.384,0.563,1.465,794,525,0.173,0.661,1585,973,0.320,0.614
4,20,Referenz,7885,3005,0.381,0.435,1.142,792,328,0.109,0.414,1579,657,0.219,0.416
5,20,Boosting,7885,3005,0.381,0.561,1.471,792,580,0.193,0.732,1579,991,0.330,0.628
6,27,Referenz,7828,2947,0.376,0.429,1.139,786,318,0.108,0.405,1567,643,0.218,0.410
7,27,Boosting,7828,2947,0.376,0.625,1.660,786,595,0.202,0.757,1567,1006,0.341,0.642
8,55,Referenz,7573,2691,0.355,0.404,1.136,760,294,0.109,0.387,1517,594,0.221,0.392
9,55,Boosting,7573,2691,0.355,0.684,1.926,760,651,0.242,0.857,1517,1080,0.401,0.712


### Art und Vorlauf der Hinweise auf 2014J

Wir betrachten bei den Hinweisen getrennt, wie viele Personen später
abbrechen und wie viele nicht bestehen. Für erkannte Abbrüche messen
wir den Abstand zwischen Hinweis und Abmeldetag.

Diese Angaben werden nur zur Auswertung nach der Vorhersage verwendet.

In [7]:
gruppen = ["stichtag", "code_module", "code_presentation"]

rangliste_test = (
    risiken_2014j
    .sort_values(
        gruppen + ["risiko"],
        ascending=[True, True, True, False],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

rangliste_test["rang_im_kurs"] = (
    rangliste_test.groupby(gruppen).cumcount()
)
kursgroesse_test = (
    rangliste_test.groupby(gruppen)["id_student"]
    .transform("size")
)

for anteil in (10, 20):
    rangliste_test[f"hinweis_{anteil}"] = (
        rangliste_test["rang_im_kurs"]
        < np.ceil(kursgroesse_test * anteil / 100)
    )

# Kontrolle gegen die zuvor berechneten Hinweiszahlen.
erwartet = (
    vergleich_2014j.loc[
        vergleich_2014j["verfahren"].eq("Boosting")
    ]
    .set_index("stichtag")["hinweise_10"]
    .sort_index()
    .to_numpy()
)
beobachtet = (
    rangliste_test.groupby("stichtag")["hinweis_10"]
    .sum()
    .sort_index()
    .to_numpy()
)
assert np.array_equal(beobachtet, erwartet)

# Ergebnis und Abmeldetag werden erst nach den Vorhersagen ergänzt.
test_details = (
    rangliste_test
    .merge(
        info_ergebnis,
        on=schluessel,
        validate="many_to_one",
    )
    .merge(
        abmeldungen,
        on=schluessel,
        validate="many_to_one",
    )
)
test_details["abmeldetag"] = pd.to_numeric(
    test_details["date_unregistration"],
    errors="coerce",
)

assert (
    test_details["final_result"].isin(["Withdrawn", "Fail"])
    == test_details["abbruch_oder_fail"]
).all()

zeilen = []

for tag in sorted(test_details["stichtag"].unique()):
    am_tag = test_details.loc[
        test_details["stichtag"].eq(tag)
    ]

    for anteil in (10, 20):
        hinweise = am_tag.loc[
            am_tag[f"hinweis_{anteil}"]
        ]
        abbruch = hinweise.loc[
            hinweise["final_result"].eq("Withdrawn")
        ]
        fail = hinweise.loc[
            hinweise["final_result"].eq("Fail")
        ]

        vorlauf = abbruch["abmeldetag"] - tag
        innerhalb_14 = int(vorlauf.le(14).sum())

        zeilen.append({
            "stichtag": tag,
            "hinweisquote": anteil,
            "hinweise": len(hinweise),
            "abbruch_treffer": len(abbruch),
            "fail_treffer": len(fail),
            "median_vorlauf_abbruch_tage": round(
                vorlauf.median(), 1
            ),
            "abbruch_innerhalb_14_tagen": innerhalb_14,
            "anteil_innerhalb_14_prozent": round(
                100 * innerhalb_14 / len(abbruch), 1
            ) if len(abbruch) else np.nan,
        })

vorlauf_2014j = pd.DataFrame(zeilen)
display(vorlauf_2014j)

,stichtag,hinweisquote,hinweise,abbruch_treffer,fail_treffer,median_vorlauf_abbruch_tage,abbruch_innerhalb_14_tagen,anteil_innerhalb_14_prozent
0,6,10,867,424,205,6.0,256,60.4
1,6,20,1731,735,432,6.0,418,56.9
2,13,10,794,223,302,62.0,25,11.2
3,13,20,1585,438,535,67.5,43,9.8
4,20,10,792,248,332,53.5,41,16.5
5,20,20,1579,431,560,61.0,61,14.2
6,27,10,786,265,330,48.0,36,13.6
7,27,20,1567,435,571,68.0,50,11.5
8,55,10,760,227,424,41.0,64,28.2
9,55,20,1517,397,683,53.0,86,21.7


## Wöchentliche Hinweise auf 2014J simulieren

Für die 16 Wochen von Tag 6 bis Tag 111 trainieren wir jeweils ein
Boosting-Modell auf früheren Kurspräsentationen. Die bereits festgelegte
Trennung nach Personen gilt weiterhin.

Pro Kurspräsentation erhalten die höchsten 10 % beziehungsweise 20 %
einen Hinweis. Wir zählen anschließend neue und wiederholte Hinweise
je Kurseintrag. Die historischen Daten enthalten keine Interventionen;
die Auswertung misst daher keine Wirkung einer Ansprache.

In [8]:
datei_voll = (
    projektordner
    / "data"
    / "processed"
    / "oulad_wochenmodellbasis_tag6_bis111.csv"
)
wochen_voll = pd.read_csv(datei_voll)

training_voll = wochen_voll.loc[
    wochen_voll["code_presentation"].ne("2014J")
].copy()

# Dieselben ausgeschlossenen Personen wie in der vorherigen
# zeitlichen Prüfung verwenden.
pruefung_voll = wochen_voll.loc[
    wochen_voll["code_presentation"].eq("2014J")
    & ~wochen_voll["id_student"].isin(fruehere_personen)
].copy()

assert set(training_voll["id_student"]).isdisjoint(
    pruefung_voll["id_student"]
)

wochen_risiken = []

for tag in sorted(wochen_voll["stichtag"].unique()):
    train = training_voll.loc[
        training_voll["stichtag"].eq(tag)
    ]
    test = pruefung_voll.loc[
        pruefung_voll["stichtag"].eq(tag)
    ]

    modell = erstelle_modell("Boosting", alle_merkmale)
    modell.fit(
        train[alle_merkmale],
        train["abbruch_oder_fail"].astype(int),
    )

    teil = test[
        schluessel + ["stichtag", "abbruch_oder_fail"]
    ].copy()
    teil["risiko"] = modell.predict_proba(
        test[alle_merkmale]
    )[:, 1]

    wochen_risiken.append(teil)
    print("Fertig: Tag", tag)

risiken_alle_wochen = pd.concat(
    wochen_risiken,
    ignore_index=True,
)

print("Datenstände in 2014J:", len(risiken_alle_wochen))
print(
    "Verschiedene Kurseinträge:",
    len(risiken_alle_wochen.drop_duplicates(schluessel)),
)

Fertig: Tag 6
Fertig: Tag 13
Fertig: Tag 20
Fertig: Tag 27
Fertig: Tag 34
Fertig: Tag 41
Fertig: Tag 48
Fertig: Tag 55
Fertig: Tag 62
Fertig: Tag 69
Fertig: Tag 76
Fertig: Tag 83
Fertig: Tag 90
Fertig: Tag 97
Fertig: Tag 104
Fertig: Tag 111
Datenstände in 2014J: 121447
Verschiedene Kurseinträge: 8682


In [9]:
gruppen = ["stichtag", "code_module", "code_presentation"]

rangliste_wochen = (
    risiken_alle_wochen
    .sort_values(
        gruppen + ["risiko"],
        ascending=[True, True, True, False],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

rangliste_wochen["rang_im_kurs"] = (
    rangliste_wochen.groupby(gruppen).cumcount()
)
kursgroesse = (
    rangliste_wochen.groupby(gruppen)["id_student"]
    .transform("size")
)

# Ein Fall zählt für den kumulativen Recall nur einmal,
# auch wenn er an mehreren Wochenenden einen Hinweis erhält.
alle_eintraege = pruefung_voll.drop_duplicates(schluessel)
faelle_im_fenster = int(
    alle_eintraege["abbruch_oder_fail"].sum()
)

wochen_ergebnisse = []

for anteil in (10, 20):
    kontingent = np.ceil(kursgroesse * anteil / 100)
    markiert = rangliste_wochen.loc[
        rangliste_wochen["rang_im_kurs"] < kontingent
    ].copy()

    # Nach dem ersten Hinweis gelten weitere Hinweise für
    # denselben Kurseintrag als Wiederholung.
    markiert = markiert.sort_values("stichtag")
    markiert["erstmalig"] = (
        markiert.groupby(schluessel).cumcount().eq(0)
    )
    markiert["neuer_treffer"] = (
        markiert["erstmalig"]
        & markiert["abbruch_oder_fail"]
    )

    pro_woche = (
        markiert.groupby("stichtag", as_index=False)
        .agg(
            hinweise=("id_student", "size"),
            neue_hinweise=("erstmalig", "sum"),
            neue_treffer=("neuer_treffer", "sum"),
        )
    )
    pro_woche["wiederholte_hinweise"] = (
        pro_woche["hinweise"]
        - pro_woche["neue_hinweise"]
    )
    pro_woche["kumulativ_erkannte_faelle"] = (
        pro_woche["neue_treffer"].cumsum()
    )
    pro_woche["kumulativer_recall"] = (
        pro_woche["kumulativ_erkannte_faelle"]
        / faelle_im_fenster
    )
    pro_woche["hinweisquote"] = anteil

    wochen_ergebnisse.append(pro_woche)

wochenverlauf = pd.concat(
    wochen_ergebnisse,
    ignore_index=True,
)

print("Verschiedene Kurseinträge im Fenster:", len(alle_eintraege))
print("Davon spätere Fälle:", faelle_im_fenster)
display(wochenverlauf.round(3))

Verschiedene Kurseinträge im Fenster: 8682
Davon spätere Fälle: 3800


,stichtag,hinweise,neue_hinweise,neue_treffer,wiederholte_hinweise,kumulativ_erkannte_faelle,kumulativer_recall,hinweisquote
0,6,867,867,629,0,629,0.166,10
1,13,794,507,312,287,941,0.248,10
2,20,792,361,242,431,1183,0.311,10
3,27,786,177,122,609,1305,0.343,10
4,34,778,146,100,632,1405,0.370,10
5,41,775,113,78,662,1483,0.390,10
6,48,769,65,36,704,1519,0.400,10
7,55,760,141,106,619,1625,0.428,10
8,62,754,106,83,648,1708,0.449,10
9,69,746,60,51,686,1759,0.463,10


### Erster Hinweis je Kurseintrag

Für jeden Kurseintrag zählt nur der erste Hinweis. Wir prüfen getrennt,
wie viele spätere Abbrüche und Fälle von Nichtbestehen damit erreicht
wurden. Bei Abbrüchen messen wir die Tage vom ersten Hinweis bis zur
Abmeldung.

Wiederholte Hinweise werden nicht erneut als erkannter Fall gezählt.

In [10]:
# Grundgesamtheit: alle Kurseinträge, die mindestens an einem
# der 16 Stichtage für einen Hinweis infrage kamen.
alle_eintraege = (
    pruefung_voll[schluessel]
    .drop_duplicates()
    .merge(
        info_ergebnis,
        on=schluessel,
        validate="one_to_one",
    )
)

alle_abbrueche = int(
    alle_eintraege["final_result"].eq("Withdrawn").sum()
)
alle_fail = int(
    alle_eintraege["final_result"].eq("Fail").sum()
)
alle_faelle = alle_abbrueche + alle_fail

gruppen = ["stichtag", "code_module", "code_presentation"]
kursgroesse_wochen = (
    rangliste_wochen.groupby(gruppen)["id_student"]
    .transform("size")
)

erster_hinweis_ergebnisse = []

for anteil in (10, 20):
    # Zunächst alle Wochenhinweise beim gewählten Budget auswählen.
    kontingent = np.ceil(
        kursgroesse_wochen * anteil / 100
    )
    mit_hinweis = rangliste_wochen.loc[
        rangliste_wochen["rang_im_kurs"] < kontingent
    ]

    # Die früheste markierte Woche ist der erste Hinweis
    # für diesen konkreten Kurseintrag.
    zuerst = (
        mit_hinweis
        .sort_values("stichtag")
        .drop_duplicates(schluessel)
        .merge(
            info_ergebnis,
            on=schluessel,
            validate="one_to_one",
        )
        .merge(
            abmeldungen,
            on=schluessel,
            validate="one_to_one",
        )
    )

    abbruch = zuerst.loc[
        zuerst["final_result"].eq("Withdrawn")
    ]
    fail = zuerst.loc[
        zuerst["final_result"].eq("Fail")
    ]

    abmeldetag = pd.to_numeric(
        abbruch["date_unregistration"],
        errors="coerce",
    )
    vorlauf = abmeldetag - abbruch["stichtag"]

    assert vorlauf.notna().all()
    assert vorlauf.gt(0).all()

    erkannte_faelle = len(abbruch) + len(fail)

    erster_hinweis_ergebnisse.append({
        "hinweisquote_je_woche": anteil,
        "verschiedene_mit_hinweis": len(zuerst),
        "hinweisanteil_ueber_16_wochen": (
            len(zuerst) / len(alle_eintraege)
        ),
        "erkannte_abbrueche": len(abbruch),
        "recall_abbruch": len(abbruch) / alle_abbrueche,
        "erkannte_fail": len(fail),
        "recall_fail": len(fail) / alle_fail,
        "recall_gesamt": erkannte_faelle / alle_faelle,
        "trefferquote_ersthinweise": (
            erkannte_faelle / len(zuerst)
        ),
        "median_vorlauf_abbruch": vorlauf.median(),
        "abbrueche_binnen_7_tagen": int(
            vorlauf.le(7).sum()
        ),
        "abbrueche_binnen_14_tagen": int(
            vorlauf.le(14).sum()
        ),
    })

erster_hinweis_vergleich = pd.DataFrame(
    erster_hinweis_ergebnisse
)

print("Kurseinträge insgesamt:", len(alle_eintraege))
print("Spätere Abbrüche:", alle_abbrueche)
print("Späteres Nichtbestehen:", alle_fail)
display(erster_hinweis_vergleich.round(3))

Kurseinträge insgesamt: 8682
Spätere Abbrüche: 2184
Späteres Nichtbestehen: 1616


,hinweisquote_je_woche,verschiedene_mit_hinweis,hinweisanteil_ueber_16_wochen,erkannte_abbrueche,recall_abbruch,erkannte_fail,recall_fail,recall_gesamt,trefferquote_ersthinweise,median_vorlauf_abbruch,abbrueche_binnen_7_tagen,abbrueche_binnen_14_tagen
0,10,2718,0.313,978,0.448,906,0.561,0.496,0.693,38.0,307,355
1,20,4768,0.549,1468,0.672,1293,0.800,0.727,0.579,40.0,459,519


### Fazit der wöchentlichen Simulation

Bei einer Auswahl von 20 % je Kurs und Woche erhalten über Tag 6 bis 111
insgesamt 4.768 von 8.682 Kurseinträgen mindestens einmal einen Hinweis.
Damit werden 67,2 % der späteren Abbrüche und 80,0 % der späteren
Nichtbestehensfälle erfasst. Die Trefferquote der erstmaligen Hinweise
beträgt 57,9 %.

Für den Prototyp unterscheiden wir eine hohe Priorität (oberste 10 %)
und eine weitere Prüfliste (nächste 10 %). Wiederholte Hinweise werden
sichtbar markiert. Hinweise sind ein Anlass zur fachlichen Prüfung und
keine Diagnose oder automatische Interventionsentscheidung.

Die zeitliche Prüfung auf 2014J ist explorativ; die Wirksamkeit von
Unterstützungsmaßnahmen lässt sich mit OULAD nicht messen.